In [1]:
import numpy as np

def write_header_lines(file, layer_height, line_width, layer_count, mesh, start_x, start_y, start_z=0.3):
    with open(file, 'w') as f:
        header_lines = [
            f";layer_height = {layer_height}",
            f"\n;line_width = {line_width}",
            f"\n;layer_count = {layer_count}",
            f"\n;mesh = {mesh}"
        ]
        
        initialize_lines = [
            "\nG21 ;start of the code",
            "\nG1 Z15 F300",
            "\nG28 X0 Y0 ;Home",
            "\nG92 X0 Y0 ;Consider this as current",
            "\nG0 X50 Y50 F3000 ;Go-to Offset",  
            "\nG92 X0 Y0 ;Reset",
            "\n",
            f"\nG0 F3600 X{start_x:.3f} Y{start_y:.3f} Z{start_z:.3f} ;Go to start position",
            "\nM7",
            "\nG4 P150",
            "\n\n"
        ]
        
        f.writelines(header_lines)
        f.writelines(initialize_lines)
        
        
def writ_finish_lines(file):
    with open(file, 'a') as f:
        finish_lines = [
            "\n\n;Finish",
            "\nM9",
            "\nG1 Z10.000"
            "\nG28 X-100 Y-100;Home",
        ]
        f.writelines(finish_lines)
    
        
def write_G1_line(delta_x, delta_y, xy, lines, pause=False):
    xy += [delta_x, delta_y]
    lines.append(f"\nG1 X{xy[0]:.3f} Y{xy[1]:.3f}")
    if pause:
        lines.append(f"\nG4 P5")
    # print(xy)
    return xy, lines


def write_init_layer(xy, z, lines):
    lines.append(f"\nG1 X{xy[0]:.3f} Y{xy[1]:.3f} Z{z:.3f}")
    return lines    
       
       
def move_to(file, x, y, z):
    lines = [
        f"\nM9",
        f"\nG1 Z10.000",
        f"\nG1 X{x:.3f} Y{y:.3f}",
        f"\nG1 Z{z:.3f}",
        f"\nM7",
        f"\nG4 P150"
    ]
    with open(file, 'a') as f:
        f.writelines(lines)
       
        
def polygon_inside_move(pl, w):
    
    sl = np.array([pl[1]-pl[0]])
    for i in range (len(pl)-2):
        sl = np.vstack((sl, pl[i+2]-pl[i+1]))
    sl = np.vstack((sl, pl[0]-pl[-1]))
    
    pl_new = np.array([])

    for i in range (len(sl)):
        if i == 0:
            x1 = sl[0] / np.linalg.norm(sl[0])
            x2 = - sl[-1] / np.linalg.norm(sl[-1])
        else:
            x1 = sl[i] / np.linalg.norm(sl[i])
            x2 = - sl[i-1] / np.linalg.norm(sl[i-1])
            
        if (float(np.cross(x1, x2)) >= 0):
            theta = np.arccos(np.dot(x1, x2))/2
            if np.linalg.norm(x1 + x2) == 0:
                xm = np.array([-x1[1],x1[0]])/np.linalg.norm(x1) * w
            else:
                xm = (x1 + x2) / np.linalg.norm(x1 + x2) * w / np.sin(theta)
            if i == 0:
                pl_new = pl[i] + xm
            else:
                pl_new = np.vstack((pl_new, pl[i] + xm))
        
        elif (float(np.cross(x1, x2)) < 0):
            theta = np.arccos(np.dot(x1, x2))/2
            if np.linalg.norm(x1 + x2) == 0:
                xm = np.array([-x1[1],x1[0]])/np.linalg.norm(x1) * w
            else:
                xm = (x1 + x2) / np.linalg.norm(x1 + x2) * w / np.sin(theta)
            if i == 0:
                pl_new = pl[i] - xm
            else:
                pl_new = np.vstack((pl_new, pl[i] - xm))
    # print(pl_new)
    return pl_new


def line_intersection(line1, line2):
    """
    求两条直线的交点。

    Args:
        line1: 定义第一条直线的两个点的列表，[[x1, y1], [x2, y2]]。
        line2: 定义第二条直线的两个点的列表，[[x3, y3], [x4, y4]]。

    Returns:
        表示交点的 [x, y] 坐标列表，如果直线平行则返回 None。
    """
    # 从直线上点的坐标中提取 x、y 值
    x1, y1 = line1[0]
    x2, y2 = line1[1]
    x3, y3 = line2[0]
    x4, y4 = line2[1]

    # 计算直线的斜率和 y 轴截距
    denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)

    # 检查直线是否平行或重合
    if denominator == 0:
        return None

    # 计算交点的 x、y 坐标
    x = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denominator
    y = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denominator

    return [x, y]


def calculate_side_vector(pl):
    sv = np.array([pl[1]-pl[0]])
    for i in range (len(pl)-2):
        sv = np.vstack((sv, pl[i+2]-pl[i+1]))
    sv = np.vstack((sv, pl[0]-pl[-1]))
    return sv


def calculate_side_length(pl, sv0):
    sv = calculate_side_vector(pl)
    sl = []
    for i in range (len(sv)):
        sl.append(np.linalg.norm(sv[i]) * np.sign(np.dot(sv0[i], sv[i])))
    return sl
  

def side_move_inside(pl, ml):
    
    pn = len(pl)
    lines = np.array([])
        
    for i in range (pn):
        v = np.array([pl[(i+1)%pn][1]-pl[i][1],-pl[(i+1)%pn][0]+pl[i][0]])
        v = v/np.linalg.norm(v)*ml[i]
        if len(lines)==0:
            lines = np.array([[[pl[i][0]+v[0],pl[i][1]+v[1]],[pl[(i+1)%pn][0]+v[0],pl[(i+1)%pn][1]+v[1]]]])
        else:
            lines = np.vstack((lines, np.array([[[pl[i][0]+v[0],pl[i][1]+v[1]],[pl[(i+1)%pn][0]+v[0],pl[(i+1)%pn][1]+v[1]]]])))
        
    pl_new = np.array([])
    for i in range (pn):
        if len(pl_new)==0:
            pl_new = np.array([line_intersection(lines[i],lines[(i+1)%pn])])
        else:
            pl_new = np.vstack((pl_new, line_intersection(lines[i],lines[(i+1)%pn])))
        
    return pl_new




In [26]:
def side_move_inside(pl, ml):
    
    pn = len(pl)
    lines = np.array([])
        
    for i in range (pn):
        v = np.array([pl[(i+1)%pn][1]-pl[i][1],-pl[(i+1)%pn][0]+pl[i][0]])
        v = v/np.linalg.norm(v)*ml[i]
        if len(lines)==0:
            lines = np.array([[[pl[i][0]+v[0],pl[i][1]+v[1]],[pl[(i+1)%pn][0]+v[0],pl[(i+1)%pn][1]+v[1]]]])
        else:
            lines = np.vstack((lines, np.array([[[pl[i][0]+v[0],pl[i][1]+v[1]],[pl[(i+1)%pn][0]+v[0],pl[(i+1)%pn][1]+v[1]]]])))
        
    pl_new = []
    for i in range (pn):
        pl_new.append(line_intersection(lines[i],lines[(i+1)%pn]))
        
    return pl_new


In [163]:
def truncate_decimals(arr, decimals=3):
    """
    将 NumPy 数组中的每个元素截断到指定的小数位数。

    Args:
    arr: 输入数组。
    decimals: 要保留的小数位数。

    Returns:
    一个新的 NumPy 数组，其中每个元素都被截断到指定的小数位数。
    """

    factor = 10 ** decimals
    return np.trunc(arr * factor) / factor


def draw_layer_out_in(file, pl, z, w, extra_line):
    
    pl = polygon_inside_move(pl, w/2)
    xy = np.array([pl[0][0], pl[0][1]])
    lines = []
    lines = write_init_layer(xy, z, lines)
    
    # with open(file, 'a') as f:
    #     f.writelines(lines)
    # lines = []
    
    sv0 = calculate_side_vector(pl)
    sl = calculate_side_length(pl, sv0)
    sv = calculate_side_vector(pl)
    
    while (len(pl)>2):
        xy, lines = write_G1_line(pl[0][0]-xy[0], pl[0][1]-xy[1], xy, lines, pause=False)
        
        sv = calculate_side_vector(pl)
        for i in range (len(pl)):
            xy, lines = write_G1_line(sv[i][0], sv[i][1], xy, lines)
            
        # with open(file, 'a') as f:
        #     f.writelines(lines)
        # lines = []
            
        pl = polygon_inside_move(pl, w)
        
        while(1):
            sl = calculate_side_length(pl, sv0)
            
            dl = []
            for i in range (len(sl)):
                if sl[i] <=0:
                    dl.append(i)
                             
            for i in range (len(dl)):
                k = dl[len(dl)-i-1]
                p_new = line_intersection([pl[k-1], pl[k]], [pl[(k+1)%len(pl)], pl[(k+2)%len(pl)]])
                if p_new is not None:
                    pl[(k+1)%len(pl)] = p_new
                    pl = np.delete(pl, k, axis=0)
                    sv0 = np.delete(sv0, k, axis=0)
                else:
                    p_new_1 = line_intersection([pl[k-2], pl[k-1]], [pl[(k+1)%len(pl)], pl[(k+2)%len(pl)]])
                    p_new_2 = line_intersection([pl[k-1], pl[k]], [pl[(k+2)%len(pl)], pl[(k+3)%len(pl)]])
                    p_mid = (pl[k]+pl[(k+1)%len(pl)]) / 2
                    if np.linalg.norm(p_new_1 - p_mid) < np.linalg.norm(p_new_2 - p_mid):
                        pl[(k+1)%len(pl)] = p_new_1
                        pl = np.delete(pl, k, axis=0)
                        pl = np.delete(pl, k-1, axis=0)
                        sv0 = np.delete(sv0, k, axis=0)
                        sv0 = np.delete(sv0, k-1, axis=0)
                    elif np.linalg.norm(p_new_1 - p_mid) > np.linalg.norm(p_new_2 - p_mid):
                        pl[(k+2)%len(pl)] = p_new_2
                        pl = np.delete(pl, (k+1)%len(pl), axis=0)
                        sv0 = np.delete(sv0, (k+1)%len(pl), axis=0)
                        if (k+1)%len(pl) == k+1:
                            pl = np.delete(pl, k, axis=0)
                            sv0 = np.delete(sv0, k, axis=0)
                        else:
                            pl = np.delete(pl, k-1, axis=0)
                            sv0 = np.delete(sv0, k-1, axis=0)  
                    else:
                        k -= 1
                        if len(pl)-k >= 4:
                            for i in range (4):
                                pl = np.delete(pl, k, axis=0)
                                sv0 = np.delete(sv0, k, axis=0)
                        else:
                            j = len(pl)-k
                            for i in range (j):
                                pl = np.delete(pl, k, axis=0)
                                sv0 = np.delete(sv0, k, axis=0)
                            for i in range (4-j):
                                pl = np.delete(pl, 0, axis=0)
                                sv0 = np.delete(sv0, 0, axis=0)
                                
                if (len(pl) < 3):
                    break
                
            if (len(dl) == 0) or (len(pl) < 3):
                break

        
    with open(file, 'a') as f:
        f.writelines(lines)
    lines = []
    
    return pl
        
        
def draw_layer_in_out(file, pl, z, w, extra_line):
    
    pl = polygon_inside_move(pl, w/2)
    xy = np.array([pl[0][0], pl[0][1]])
    lines = []
    lines = write_init_layer(xy, z, lines)
    
    # with open(file, 'a') as f:
    #     f.writelines(lines)
    # lines = []
    
    sv0 = calculate_side_vector(pl)
    sl = calculate_side_length(pl, sv0)
    sv = calculate_side_vector(pl)
    
    while (len(pl)>2):
        xy, lines = write_G1_line(pl[0][0]-xy[0], pl[0][1]-xy[1], xy, lines, pause=False)
        
        sv = calculate_side_vector(pl)
        for i in range (len(pl)):
            xy, lines = write_G1_line(sv[i][0], sv[i][1], xy, lines)
            
        # with open(file, 'a') as f:
        #     f.writelines(lines)
        # lines = []
            
        pl = polygon_inside_move(pl, w)
        
        while(1):
            sl = calculate_side_length(pl, sv0)
            
            dl = []
            for i in range (len(sl)):
                if sl[i] <=0:
                    dl.append(i)
                             
            for i in range (len(dl)):
                k = dl[len(dl)-i-1]
                p_new = line_intersection([pl[k-1], pl[k]], [pl[(k+1)%len(pl)], pl[(k+2)%len(pl)]])
                if p_new is not None:
                    pl[(k+1)%len(pl)] = p_new
                    pl = np.delete(pl, k, axis=0)
                    sv0 = np.delete(sv0, k, axis=0)
                else:
                    p_new_1 = line_intersection([pl[k-2], pl[k-1]], [pl[(k+1)%len(pl)], pl[(k+2)%len(pl)]])
                    p_new_2 = line_intersection([pl[k-1], pl[k]], [pl[(k+2)%len(pl)], pl[(k+3)%len(pl)]])
                    p_mid = (pl[k]+pl[(k+1)%len(pl)]) / 2
                    if np.linalg.norm(p_new_1 - p_mid) < np.linalg.norm(p_new_2 - p_mid):
                        pl[(k+1)%len(pl)] = p_new_1
                        pl = np.delete(pl, k, axis=0)
                        pl = np.delete(pl, k-1, axis=0)
                        sv0 = np.delete(sv0, k, axis=0)
                        sv0 = np.delete(sv0, k-1, axis=0)
                    elif np.linalg.norm(p_new_1 - p_mid) > np.linalg.norm(p_new_2 - p_mid):
                        pl[(k+2)%len(pl)] = p_new_2
                        pl = np.delete(pl, (k+1)%len(pl), axis=0)
                        sv0 = np.delete(sv0, (k+1)%len(pl), axis=0)
                        if (k+1)%len(pl) == k+1:
                            pl = np.delete(pl, k, axis=0)
                            sv0 = np.delete(sv0, k, axis=0)
                        else:
                            pl = np.delete(pl, k-1, axis=0)
                            sv0 = np.delete(sv0, k-1, axis=0)  
                    else:
                        k -= 1
                        if len(pl)-k >= 4:
                            for i in range (4):
                                pl = np.delete(pl, k, axis=0)
                                sv0 = np.delete(sv0, k, axis=0)
                        else:
                            j = len(pl)-k
                            for i in range (j):
                                pl = np.delete(pl, k, axis=0)
                                sv0 = np.delete(sv0, k, axis=0)
                            for i in range (4-j):
                                pl = np.delete(pl, 0, axis=0)
                                sv0 = np.delete(sv0, 0, axis=0)
                                
                if (len(pl) < 3):
                    break
                
            if (len(dl) == 0) or (len(pl) < 3):
                break
                
    lines_write = []
    lines_write = write_init_layer(xy, z, lines_write)
    l = len(lines)
    for i in range (len(lines)):
        if (lines[i].startswith('G4')):
            lines_write.append(lines[l-i-2])
            lines_write.append(lines[l-i-1])
            i += 1
        else:
            lines_write.append(lines[l-i-1])
            
    with open(file, 'a') as f:
        f.writelines(lines_write)
    
    return pl
        
        
def draw_layer(file, point_list, z, w, direction, extra_line=0):
    """Draw single platform layer by concentrate infill

    Args:
        point_list (ndarray)  : np.array([[x1, y1], [x2, y2], ...])
        w (float)             : line width
        direction (int)       : 0 for out -> in (counter clockwise) 
                                1 for in -> out (clockwise)
    """
    
    if direction == 0:
        pl = draw_layer_out_in(file, point_list, z, w, extra_line)
    elif direction == 1:
        pl = draw_layer_in_out(file, point_list, z, w, extra_line)
        
    return pl

In [141]:
mesh = 'bird_base_old'
layer_height = 0.4
line_width = 0.4
height = 4
h = height
layer_count = int(height/layer_height)

xbl = [np.sqrt(h**2-(h-layer_height/2-i*layer_height)**2) for i in range (layer_count)]
xsl = [np.sqrt(h**2-(h-layer_height/2-(layer_count-i-1)*layer_height)**2) for i in range (layer_count)]
xbl[0] = 0

x1 = 25
x2 = 50
x3 = 40
x4 = 18
x5 = 15

y1 = 40
y2 = 15
y3 = 8
y4 = 6
y5 = 2

k = 8
g = 0.4
l = 9.6

file = '8.20/'+mesh+'.gcode'
write_header_lines(file, layer_height, line_width, layer_count, mesh, -x1,-(y3+k+g*4))

# bottom layer

# [-x1,-(y3+k+g*4)],[-x5,-(y2+k+g*4)],[-x5,-(y2+k*2+g*8+l)],[0,-(y1+k*2+g*8+l)],[x5,-(y2+k*2+g*8+l)],[x5,-(y2+k+g*4)],[x4,-(y3+k+g*4)],[x4,-y3],[x4+k+g*4,-y3],[x3+k+g*4,-y4],[x3+k*2+g*8,-y4],[x2+k*2+g*8,-y5],[x2+k*2+g*8,y5],[x3+k*2+g*8,y4],[x3+k+g*4,y4],[x4+k+g*4,y3],[x4,y3],[x4,y3+k+g*4],[x5,y2+k+g*4],[x5,y2+k*2+g*8+l],[0,y1+k*2+g*8+l],[-x5,y2+k*2+g*8+l],[-x5,y2+k+g*4],[-x1,y3+k+g*4]

p1 = np.array([-x1,-(y3+k+g*4)])
p2 = np.array([-x5,-(y2+k+g*4)])
p3 = np.array([-x5+0.1,-(y2+k*2+g*12+l)])
p4 = np.array([0,-(y1+k*2+g*12+l)])
p5 = np.array([x5-0.1,-(y2+k*2+g*12+l)])
p6 = np.array([x5,-(y2+k+g*4)])
p7 = np.array([x4,-(y3+k+g*4)])
p8 = np.array([x4,-y3])
p9 = np.array([x4+k+g*4,-y3])
p10 = np.array([x3+k+g*4,-y4-0.1])
p11 = np.array([x3+k*2+g*8,-y4])
p12 = np.array([x2+k*2+g*8,-y5])
p13 = np.array([x2+k*2+g*8,y5])
p14 = np.array([x3+k*2+g*8,y4])
p15 = np.array([x3+k+g*4,y4+0.1])
p16 = np.array([x4+k+g*4,y3])
p17 = np.array([x4,y3])
p18 = np.array([x4,y3+k+g*4])
p19 = np.array([x5,y2+k+g*4])
p20 = np.array([x5-0.1,y2+k*2+g*12+l])
p21 = np.array([0,y1+k*2+g*12+l])
p22 = np.array([-x5+0.1,y2+k*2+g*12+l])
p23 = np.array([-x5,y2+k+g*4])
p24 = np.array([-x1,y3+k+g*4])

# pl = np.array([p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24])

# _ = draw_layer(file, pl, 0.3, line_width, 0)
# _ = draw_layer(file, pl, 0.7, line_width, 1)

# body
# pl = np.array([[-x1,-y3-k/2-g],[x4,-y3-k/2-g],[x4,-y3],[x4+k/2+g,-y3],[x4+k/2+g,y3],[x4,y3],[x4,y3+k/2+g],[-x1,y3+k/2+g]])
# for i in range (3):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([-xb,0,0,-xb,0,0,-xb,0])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     pl_new[0][0],pl_new[5][0] = pl_new[1][0],pl_new[1][0]
#     if i == 0:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1)
#     if i == 1:
#         print(pl_new)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([[-x1,-y3-k/2-g],[-x1+10,-y3-k/2-g],[-x1+10,y3+k/2+g],[-x1,y3+k/2+g]])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([-xb,0,-xb,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([[-x1+11.6,-y3-k/2-g],[x4-11.6,-y3-k/2-g],[x4-11.6,-0.8],[-x1+11.6,-0.8]])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([-xb,0,0,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([[-x1+11.6,0.8],[x4-11.6,0.8],[x4-11.6,y3+k/2+g],[-x1+11.6,y3+k/2+g]])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,-xb,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([[x4-10,-y3-k/2-g],[x4,-y3-k/2-g],[x4,-y3],[x4+k/2+g,-y3],[x4+k/2+g,-0.8],[x4-10,-0.8]])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([-xb,0,0,-xb,0,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([[x4-10,0.8],[x4+k/2+g,0.8],[x4+k/2+g,y3],[x4,y3],[x4,y3+k/2+g],[x4-10,y3+k/2+g]])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,0,-xb,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
    
# # side
# pl = np.array([p1+np.array([0,k/2+g]),p1,p2,p2-np.array([0,k/2+g]),p6-np.array([0,k/2+g]),p6,p7,p7+np.array([0,k/2+g])])
# for i in range (layer_count//2):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([0,0,0,-xs,0,0,0,-xb])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     if i == 0:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# pl = np.array([p24,p24-np.array([0,k/2+g]),p18-np.array([0,k/2+g]),p18,p19,p19+np.array([0,k/2+g]),p23+np.array([0,k/2+g]),p23])
# for i in range (layer_count//2):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([0,-xb,0,0,0,-xs,0,0])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     if i == 0:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)

# # neck
# pl = np.array([p9-np.array([k/2+g,0]),p9,p10,p10+np.array([k/2+g,0]),p15+np.array([k/2+g,0]),p15,p16,p16-np.array([k/2+g,0])])
# for i in range (3):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([0,0,0,-xs,0,0,0,-xb])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     if i == 0:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# pl = np.array([p9-np.array([k/2+g,0]),p9,p10,p10+np.array([k/2+g,0]),p10+np.array([k/2+g,-0.8-p10[1]]),p9-np.array([k/2+g,0.8+p9[1]])])
# for i in range (3,5):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([0,0,0,-xs,0,-xb])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     if i == 3:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# pl = np.array([p16-np.array([k/2+g,p16[1]-0.8]),p15+np.array([k/2+g,0.8-p15[1]]),p15+np.array([k/2+g,0]),p15,p16,p16-np.array([k/2+g,0])])
# for i in range (3,5):
#     xb = xbl[i]
#     xs = xsl[i]
#     ml = np.array([0,-xs,0,0,0,-xb])
#     pl_new = side_move_inside(pl,ml)
#     pl_new = truncate_decimals(pl_new)
#     if i == 3:
#         move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
#     _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    

In [134]:
mesh = 'bird_top_old'
layer_height = 0.4
line_width = 0.4
height = 4
h = height
layer_count = int(height/layer_height)

# xbl = [np.sqrt(h**2-(h-layer_height/2-i*layer_height)**2) for i in range (layer_count)]
# xsl = [np.sqrt(h**2-(h-layer_height/2-(layer_count-i-1)*layer_height)**2) for i in range (layer_count)]
# xbl[0] = 0

# x1 = 20
# x2 = 40
# x3 = 32
# x4 = 15
# x5 = 10

# y1 = 30
# y2 = 12
# y3 = 8
# y4 = 6
# y5 = 2

# k = 8
# g = 0.4
# l = 9.6

file = '8.20/'+mesh+'.gcode'
write_header_lines(file, layer_height, line_width, layer_count, mesh, -x1,-(y3+k+g*4))

# bottom layer

# p1 = np.array([-x1,-(y3+k+g*4)])
# p2 = np.array([-x5,-(y2+k+g*4)])
# p3 = np.array([-x5+0.1,-(y2+k*2+g*8+l)])
# p4 = np.array([0,-(y1+k*2+g*8+l)])
# p5 = np.array([x5-0.1,-(y2+k*2+g*8+l)])
# p6 = np.array([x5,-(y2+k+g*4)])
# p7 = np.array([x4,-(y3+k+g*4)])
# p8 = np.array([x4,-y3])
# p9 = np.array([x4+k+g*4,-y3])
# p10 = np.array([x3+k+g*4,-y4-0.1])
# p11 = np.array([x3+k*2+g*8,-y4])
# p12 = np.array([x2+k*2+g*8,-y5])
# p13 = np.array([x2+k*2+g*8,y5])
# p14 = np.array([x3+k*2+g*8,y4])
# p15 = np.array([x3+k+g*4,y4+0.1])
# p16 = np.array([x4+k+g*4,y3])
# p17 = np.array([x4,y3])
# p18 = np.array([x4,y3+k+g*4])
# p19 = np.array([x5,y2+k+g*4])
# p20 = np.array([x5-0.1,y2+k*2+g*8+l])
# p21 = np.array([0,y1+k*2+g*8+l])
# p22 = np.array([-x5+0.1,y2+k*2+g*8+l])
# p23 = np.array([-x5,y2+k+g*4])
# p24 = np.array([-x1,y3+k+g*4])

pl = np.array([p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p20,p21,p22,p23,p24])

_ = draw_layer(file, pl, 0.3, line_width, 0)
_ = draw_layer(file, pl, 0.7, line_width, 1)

# body
pl = np.array([[-x1,-y3-k/2-g],[x4,-y3-k/2-g],[x4,-y3],[x4+k/2+g,-y3],[x4+k/2+g,y3],[x4,y3],[x4,y3+k/2+g],[-x1,y3+k/2+g]])
for i in range (3):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([-xs,0,0,-xs,0,0,-xs,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    pl_new[0][0],pl_new[5][0] = pl_new[1][0],pl_new[1][0]
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    if i == 1:
        print(pl_new)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0) 
    
# side
pl = np.array([p1+np.array([0,k/2+g]),p1,p2,p2-np.array([0,k/2+g]),p6-np.array([0,k/2+g]),p6,p7,p7+np.array([0,k/2+g])])
for i in range (layer_count//2):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,-xb,0,0,0,-xs])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p24,p24-np.array([0,k/2+g]),p18-np.array([0,k/2+g]),p18,p19,p19+np.array([0,k/2+g]),p23+np.array([0,k/2+g]),p23])
for i in range (layer_count//2):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xs,0,0,0,-xb,0,0])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)

# neck
pl = np.array([p9-np.array([k/2+g,0]),p9,p10,p10+np.array([k/2+g,0]),p15+np.array([k/2+g,0]),p15,p16,p16-np.array([k/2+g,0])])
for i in range (3):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,-xb,0,0,0,-xs])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p9-np.array([k/2+g,0]),p9,p10,p10+np.array([k/2+g,0]),p10+np.array([k/2+g,-0.8-p10[1]]),p9-np.array([k/2+g,0.8+p9[1]])])
for i in range (3,5):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,-xb,0,-xs])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p16-np.array([k/2+g,p16[1]-0.8]),p15+np.array([k/2+g,0.8-p15[1]]),p15+np.array([k/2+g,0]),p15,p16,p16-np.array([k/2+g,0])])
for i in range (3,5):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,0,0,-xs])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# head
pl = np.array([p11-np.array([k/2+g,0]),p11,p12,p13,p14,p14-np.array([k/2+g,0])])
for i in range (layer_count):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,0,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# wings
pl = np.array([p3+np.array([0,k/2+g]),p3,p4,p5,p5+np.array([0,k/2+g])])
for i in range (layer_count):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p20-np.array([0,k/2+g]),p20,p21,p22,p22-np.array([0,k/2+g])])
for i in range (layer_count):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,0,0,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
# wing middle
pl = np.array([p2-np.array([0,k/2+g]),p3+np.array([0,k/2+g]),p5+np.array([0,k/2+g]),p6-np.array([0,k/2+g])])
for i in range (3):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p2-np.array([0,k/2+g]),p3+np.array([0,k/2+g]),p3+np.array([x5-0.8,k/2+g]),p2-np.array([-x5+0.8,k/2+g])])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p6-np.array([x5-0.8,k/2+g]),p5+np.array([-x5+0.8,k/2+g]),p5+np.array([0,k/2+g]),p6-np.array([0,k/2+g])])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p2-np.array([0,k/2+g]),p3+np.array([0,k/2+g]),p5+np.array([0,k/2+g]),p6-np.array([0,k/2+g])])
for i in range (7,10):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 7:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)

pl = np.array([p19+np.array([0,k/2+g]),p20-np.array([0,k/2+g]),p22-np.array([0,k/2+g]),p23+np.array([0,k/2+g])])
for i in range (layer_count):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 0:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p19+np.array([0,k/2+g]),p20-np.array([0,k/2+g]),p20-np.array([x5-0.8,k/2+g]),p19+np.array([-x5+0.8,k/2+g])])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p23+np.array([x5-0.8,k/2+g]),p22-np.array([-x5+0.8,k/2+g]),p22-np.array([0,k/2+g]),p23+np.array([0,k/2+g])])
for i in range (3,7):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 3:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)
    
pl = np.array([p19+np.array([0,k/2+g]),p20-np.array([0,k/2+g]),p22-np.array([0,k/2+g]),p23+np.array([0,k/2+g])])
for i in range (7,10):
    xb = xbl[i]
    xs = xsl[i]
    ml = np.array([0,-xb,0,-xb])
    pl_new = side_move_inside(pl,ml)
    pl_new = truncate_decimals(pl_new)
    if i == 7:
        move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
    _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)


[[ 14.999  -8.445]
 [ 14.999  -8.   ]
 [ 15.445  -8.   ]
 [ 15.445   8.   ]
 [ 14.999   8.   ]
 [ 14.999   8.445]
 [-20.      8.445]
 [-20.     -8.445]]


In [167]:
layer_height = 0.4
line_width = 0.4
height = 4
h = height
layer_count = int(height/layer_height)

xbl = [np.sqrt(h**2-(h-layer_height/2-i*layer_height)**2) for i in range (layer_count)]
xsl = [np.sqrt(h**2-(h-layer_height/2-(layer_count-i-1)*layer_height)**2) for i in range (layer_count)]
xbl[0] = 0.4

a1 = 20
a2 = 20
a3 = 15
a4 = 10
a5 = 15

b1 = 12
b2 = 8
b3 = 20
b4 = 8
b5 = 6

k = 4
g = 0.4
l = 4.8
m = 2

p1  = np.array([-a1,-(b1+k)])
p2  = np.array([-a1,-(b1+k+g*2)])
p3  = np.array([-a1,-(b1+k*2+g*2+m)])
p4  = np.array([-a5-0.1,-(b1+k*2+g*2+b2-m)])
p5  = np.array([-a5,-(b1+k*2+g*2+b2+k)])
p6  = np.array([-a5,-(b1+k*2+g*2+b2+k+g*2)])
p7  = np.array([-a5,-(b1+k*2+g*2+b2+k+l*2+g*2)])
p8  = np.array([-a5,-(b1+k*2+g*2+b2+k+l*2+g*2*2)])
p9  = np.array([-a5,-(b1+k*2+g*2+b2+k*2+l*2+g*2*2+m)])
p10 = np.array([ 0,-(b1+k*2+g*2+b2+k*2+l*2+g*2*2+b3)])
p11 = np.array([ a5,-(b1+k*2+g*2+b2+k*2+l*2+g*2*2+m)])
p12 = np.array([ a5,-(b1+k*2+g*2+b2+k+l*2+g*2*2)])
p13 = np.array([ a5,-(b1+k*2+g*2+b2+k+l*2+g*2)])
p14 = np.array([ a5,-(b1+k*2+g*2+b2+k+g*2)])
p15 = np.array([ a5,-(b1+k*2+g*2+b2+k)])
p16 = np.array([ a5+0.1,-(b1+k*2+g*2+b2-m)])
p17 = np.array([ a2,-(b1+k*2+g*2+m)])
p18 = np.array([ a2,-(b1+k+g*2)])
p19 = np.array([ a2,-(b1+k)])
p20 = np.array([ a2,-(b1-m)])
p21 = np.array([ a2+m+k,-(b1-m)])
p22 = np.array([ a2+m+k+g*2,-(b1-m)])
p23 = np.array([ a2+m+k*2+g*2+m,-(b1-m)])
p24 = np.array([ a2+m+k*2+g*2+a3-m,-b4])
p25 = np.array([ a2+m+k*2+g*2+a3+k,-b4])
p26 = np.array([ a2+m+k*2+g*2+a3+k+g*2,-b4])
p27 = np.array([ a2+m+k*2+g*2+a3+k*2+g*2+m,-b4])
p28 = np.array([ a2+m+k*2+g*2+a3+k*2+g*2+a4,-b5])
p29 = np.array([p28[0],-p28[1]])
p30 = np.array([p27[0],-p27[1]])
p31 = np.array([p26[0],-p26[1]])
p32 = np.array([p25[0],-p25[1]])
p33 = np.array([p24[0],-p24[1]])
p34 = np.array([p23[0],-p23[1]])
p35 = np.array([p22[0],-p22[1]])
p36 = np.array([p21[0],-p21[1]])
p37 = np.array([p20[0],-p20[1]])
p38 = np.array([p19[0],-p19[1]])
p39 = np.array([p18[0],-p18[1]])
p40 = np.array([p17[0],-p17[1]])
p41 = np.array([p16[0],-p16[1]])
p42 = np.array([p15[0],-p15[1]])
p43 = np.array([p14[0],-p14[1]])
p44 = np.array([p13[0],-p13[1]])
p45 = np.array([p12[0],-p12[1]])
p46 = np.array([p11[0],-p11[1]])
p47 = np.array([p10[0],-p10[1]])
p48 = np.array([p9[0],-p9[1]])
p49 = np.array([p8[0],-p8[1]])
p50 = np.array([p7[0],-p7[1]])
p51 = np.array([p6[0],-p6[1]])
p52 = np.array([p5[0],-p5[1]])
p53 = np.array([p4[0],-p4[1]])
p54 = np.array([p3[0],-p3[1]])
p55 = np.array([p2[0],-p2[1]])
p56 = np.array([p1[0],-p1[1]])

In [176]:
def draw_platform(file, pl, ms, begin, end, layer_height=0.4, line_width=0.4):
    for i in range (begin, end):
        xb = xbl[i]
        xs = xsl[i]
        ml = np.array([])
        for j in range (len(pl)):
            if ms[j] == 0:
                ml = np.append(ml,0)
            elif ms[j] == 1:
                ml = np.append(ml,-xb)
            elif ms[j] == -1:
                ml = np.append(ml,-xs)
        pl_new = side_move_inside(pl,ml)
        pl_new = truncate_decimals(pl_new)
        # pl_new[0][0],pl_new[5][0] = pl_new[1][0],pl_new[1][0]
        if i == begin:
            move_to(file,pl_new[0][0],pl_new[0][1],1.1+i*layer_height)
        # print(i,1.1+i*layer_height)
        _ = draw_layer(file, pl_new, 1.1+i*layer_height, line_width, 0)

In [187]:
mesh = 'bird_base'


file = '8.20/'+mesh+'.gcode'
write_header_lines(file, layer_height, line_width, layer_count, mesh, p3[0],p3[1])

_ = draw_layer(file, np.array([p1,p19,p20,p21,p36,p37,p38,p56]), 0.3, line_width, 0)
_ = draw_layer(file, np.array([p2,p3,p4,p9,p10,p11,p16,p17,p18]), 0.3, line_width, 0)
_ = draw_layer(file, np.array([p55,p39,p40,p41,p46,p47,p48,p53,p54]), 0.3, line_width, 0)
_ = draw_layer(file, np.array([p22,p23,p24,p27,p28,p29,p30,p33,p34,p35]), 0.3, line_width, 0)

pl = np.array([p3,p4,p9,p10,p11,p16,p17,p20,p23,p24,p27,p28,p29,p30,p33,p34,p37,p40,p41,p46,p47,p48,p53,p54])
_ = draw_layer(file, pl, 0.7, line_width, 1)

# body
pl = np.array([p1,p19,p20,p21,p36,p37,p38,p56])
ms = [1,0,0,1,0,0,1,0]
draw_platform(file, pl, ms, 0, 3)

pl = np.array([p1,p1+np.array([10,0]),p56+np.array([10,0]),p56])
ms = [1,0,1,0]
draw_platform(file, pl, ms, 3, 7)

pl = np.array([p1+np.array([11.6,0]),p19-np.array([11.6,0]),p19-np.array([11.6,p19[1]+0.8]),p1+np.array([11.6,-p1[1]-0.8])])
ms = [1,0,0,0]
draw_platform(file, pl, ms, 3, 7)

pl = np.array([p38-np.array([11.6,0]),p56+np.array([11.6,0]),p56+np.array([11.6,-p56[1]+0.8]),p38-np.array([11.6,p38[1]-0.8])])
ms = [1,0,0,0]
draw_platform(file, pl, ms, 3, 7)

pl = np.array([np.array([p19[0]-10,-0.8]),p19-np.array([10,0]),p19,p20,p21,p21-np.array([0,p21[1]+0.8])])
ms = [0,1,0,0,1,0]
draw_platform(file, pl, ms, 3, 7)

pl = np.array([np.array([p38[0]-10,0.8]),np.array([p36[0],0.8]),p36,p37,p38,p38-np.array([10,0])])
ms = [0,1,0,0,1,0]
draw_platform(file, pl, ms, 3, 7)  
    
# sides
pl = np.array([p2,p3,p4,p5,p15,p16,p17,p18])
ms = [0,0,0,-1,0,0,0,1]
draw_platform(file, pl, ms, 0, 5)

pl = np.array([p39,p40,p41,p42,p52,p53,p54,p55])
ms = [0,0,0,-1,0,0,0,1]
draw_platform(file, pl, ms, 0, 5)

# wings
pl = np.array([p8,p9,p10,p11,p12])
ms = [0,0,0,0,-1]
draw_platform(file, pl, ms, 0, 5)

pl = np.array([p45,p46,p47,p48,p49])
ms = [0,0,0,0,-1]
draw_platform(file, pl, ms, 0, 5)

# neck
pl = np.array([p22,p23,p24,p25,p32,p33,p34,p35])
ms = [0,0,0,-1,0,0,0,1]
draw_platform(file, pl, ms, 0, 3)

pl = np.array([p22,p23,p24,p25,np.array([p25[0],-0.8]),np.array([p22[0],-0.8])])
ms = [0,0,0,-1,0,1]
draw_platform(file, pl, ms, 3, 7)

pl = np.array([p32,p33,p34,p35,np.array([p35[0],0.8]),np.array([p32[0],0.8])])
ms = [0,0,0,1,0,-1]
draw_platform(file, pl, ms, 3, 7)

# head
pl = np.array([p26,p27,p28,p29,p30,p31])
ms = [0,0,0,0,0,-1]
draw_platform(file, pl, ms, 0, 5)

# wings middle
pl = np.array([p8,p13,p14,p6])
ms = [-1,0,-1,0]
draw_platform(file, pl, ms, 0, 3)

pl = np.array([p44,p50,p51,p43])
ms = [-1,0,-1,0]
draw_platform(file, pl, ms, 0, 3)

In [180]:
type(p21[1])

numpy.int32